In [1]:
from google.oauth2 import service_account
from googleapiclient.discovery import build
import logging
import time
from typing import List, Tuple, Optional

class GoogleSheetRowMover:
    def __init__(self, credentials_file: str, batch_size: int = 20):
        # Set up logging
        logging.basicConfig(
            level=logging.INFO,
            format='%(asctime)s - %(levelname)s - %(message)s',
            handlers=[
                logging.FileHandler('sheet_mover.log'),
                logging.StreamHandler()
            ]
        )
        self.logger = logging.getLogger(__name__)
        self.batch_size = batch_size
        
        # Initialize the Google Sheets API
        scopes = ['https://www.googleapis.com/auth/spreadsheets']
        self.creds = service_account.Credentials.from_service_account_file(
            credentials_file, scopes=scopes)
        self.service = build('sheets', 'v4', credentials=self.creds)
    
    def extract_spreadsheet_id(self, sheet_url: str) -> str:
        """Extract the spreadsheet ID from a Google Sheets URL"""
        import re
        pattern = r'/spreadsheets/d/([a-zA-Z0-9-_]+)'
        match = re.search(pattern, sheet_url)
        if match:
            return match.group(1)
        raise ValueError("Invalid Google Sheet URL")
        
    def list_all_sheets(self, spreadsheet_id: str):
        """List all sheets in the spreadsheet with their IDs"""
        sheet_metadata = self.service.spreadsheets().get(spreadsheetId=spreadsheet_id).execute()
        sheets = sheet_metadata.get('sheets', '')
        self.logger.info("Available sheets in this spreadsheet:")
        for sheet in sheets:
            title = sheet['properties']['title']
            sheet_id = sheet['properties']['sheetId']
            self.logger.info(f"  - '{title}' (ID: {sheet_id})")
        return sheets
    
    def get_sheet_data(self, spreadsheet_id: str, sheet_range: str):
        """Get data from a specific range in a sheet"""
        result = self.service.spreadsheets().values().get(
            spreadsheetId=spreadsheet_id, range=sheet_range).execute()
        return result.get('values', [])
    
    def append_rows_batch(self, spreadsheet_id: str, sheet_name: str, rows_data: List[List]):
        """Append multiple rows to the specified sheet in a single batch operation"""
        if not rows_data:
            return None
            
        range_name = f"{sheet_name}!A:Z"
        body = {
            'values': rows_data
        }
        result = self.service.spreadsheets().values().append(
            spreadsheetId=spreadsheet_id, 
            range=range_name,
            valueInputOption='RAW', 
            insertDataOption='INSERT_ROWS', 
            body=body
        ).execute()
        
        self.logger.info(f"Batch appended {len(rows_data)} rows to {sheet_name}")
        return result
    
    def delete_rows_batch(self, spreadsheet_id: str, sheet_id: int, row_indices: List[int]):
        """Delete multiple rows from the sheet in a single batch operation"""
        if not row_indices:
            return None
            
        # Sort indices in descending order to avoid index shifting issues
        sorted_indices = sorted(row_indices, reverse=True)
        
        # Group consecutive indices for more efficient deletion
        delete_ranges = []
        i = 0
        while i < len(sorted_indices):
            start_idx = sorted_indices[i]
            end_idx = start_idx
            
            # Find consecutive indices
            j = i + 1
            while j < len(sorted_indices) and sorted_indices[j] == sorted_indices[j-1] - 1:
                end_idx = sorted_indices[j]
                j += 1
            
            # Add the range (swap start and end since we're going in reverse)
            delete_ranges.append({
                "deleteDimension": {
                    "range": {
                        "sheetId": sheet_id,
                        "dimension": "ROWS",
                        "startIndex": end_idx - 1,  # 0-indexed
                        "endIndex": start_idx  # exclusive end index
                    }
                }
            })
            
            i = j
        
        batch_update_request = {
            "requests": delete_ranges
        }
        
        result = self.service.spreadsheets().batchUpdate(
            spreadsheetId=spreadsheet_id, 
            body=batch_update_request
        ).execute()
        
        self.logger.info(f"Batch deleted {len(row_indices)} rows in {len(delete_ranges)} ranges")
        return result
    
    def get_sheet_id(self, spreadsheet_id: str, sheet_name: str):
        """Get the sheet ID by name"""
        sheet_metadata = self.service.spreadsheets().get(spreadsheetId=spreadsheet_id).execute()
        sheets = sheet_metadata.get('sheets', '')
        for sheet in sheets:
            if sheet['properties']['title'] == sheet_name:
                sheet_id = sheet['properties']['sheetId']
                self.logger.info(f"Found sheet ID for '{sheet_name}': {sheet_id} (type: {type(sheet_id)})")
                return sheet_id
        self.logger.error(f"Sheet '{sheet_name}' not found. Available sheets: {[s['properties']['title'] for s in sheets]}")
        return None
    
    def find_rows_with_value(self, data, column_index: int, value_to_find: str):
        """Find all rows containing the specified value in the given column"""
        matching_rows = []
        for i, row in enumerate(data):
            if len(row) > column_index and row[column_index] == value_to_find:
                matching_rows.append((i + 1, row))  # +1 because Google Sheets is 1-indexed
        return matching_rows
    
    def find_row_by_email(self, data, email_column_index: int, email_to_find: str):
        """Find the row containing the specified email"""
        for i, row in enumerate(data):
            if len(row) > email_column_index and row[email_column_index] == email_to_find:
                return (i + 1, row)  # +1 because Google Sheets is 1-indexed
        return None
    
    def clear_and_rewrite_sheet(self, spreadsheet_id: str, sheet_name: str, data: List[List]):
        """Clear a sheet and rewrite with new data"""
        # Clear the sheet first
        range_name = f"{sheet_name}!A:Z"
        self.service.spreadsheets().values().clear(
            spreadsheetId=spreadsheet_id,
            range=range_name
        ).execute()
        
        # Write the new data if there is any
        if data:
            body = {
                'values': data
            }
            result = self.service.spreadsheets().values().update(
                spreadsheetId=spreadsheet_id,
                range=f"{sheet_name}!A1",
                valueInputOption='RAW',
                body=body
            ).execute()
            self.logger.info(f"Rewrote {sheet_name} with {len(data)} rows")
            return result
        else:
            self.logger.info(f"Cleared {sheet_name} (no data to write)")
            return None
    
    def process_sheets(self, spreadsheet_id: str, target_column_index: int, email_column_index: int, 
                     source_sheet_name: str = "Sheet2", target_sheet_name: str = "Sheet3",
                     source_match_sheet_name: str = "Sheet1", target_match_sheet_name: str = "Sheet4",
                     target_value: str = "NONSENSICAL DATA POINT"):
        """
        Find rows in source_sheet where target_column contains target_value,
        move them to target_sheet, and move matching rows from source_match_sheet to target_match_sheet
        using batch processing to avoid rate limiting
        """
        try:
            # List all available sheets first
            sheets = self.list_all_sheets(spreadsheet_id)
            
            # Get sheet IDs for deletion operations
            source_sheet_id = self.get_sheet_id(spreadsheet_id, source_sheet_name)
            source_match_sheet_id = self.get_sheet_id(spreadsheet_id, source_match_sheet_name)
            
            if source_sheet_id is None:
                self.logger.error(f"Could not find '{source_sheet_name}' ID")
                return False
                
            if source_match_sheet_id is None:
                self.logger.error(f"Could not find '{source_match_sheet_name}' ID")
                return False
                
            self.logger.info(f"Using sheet IDs: {source_sheet_name}={source_sheet_id}, {source_match_sheet_name}={source_match_sheet_id}")
            
            # Get all data from source sheet
            source_data = self.get_sheet_data(spreadsheet_id, f"{source_sheet_name}!A:Z")
            
            if not source_data:
                self.logger.error(f"No data found in {source_sheet_name}")
                return False
            
            self.logger.info(f"Total rows in {source_sheet_name}: {len(source_data)}")
            
            # Get all data from source match sheet
            source_match_data = self.get_sheet_data(spreadsheet_id, f"{source_match_sheet_name}!A:Z")
            
            if not source_match_data:
                self.logger.error(f"No data found in {source_match_sheet_name}")
                return False
            
            self.logger.info(f"Total rows in {source_match_sheet_name}: {len(source_match_data)}")
            
            # Separate rows to keep and rows to move
            rows_to_move = []
            rows_to_keep = []
            
            for i, row in enumerate(source_data):
                if len(row) > target_column_index and row[target_column_index] == target_value:
                    rows_to_move.append(row)
                else:
                    rows_to_keep.append(row)
            
            self.logger.info(f"Found {len(rows_to_move)} rows to move, {len(rows_to_keep)} rows to keep in {source_sheet_name}")
            
            if not rows_to_move:
                self.logger.info(f"No rows found in {source_sheet_name} with '{target_value}' in column {target_column_index+1}")
                return True
            
            # Process matching rows from source_match_sheet
            match_rows_to_move = []
            match_rows_to_keep = []
            emails_to_move = set()
            
            # Collect emails from rows to move
            for row in rows_to_move:
                if len(row) > email_column_index:
                    emails_to_move.add(row[email_column_index])
            
            self.logger.info(f"Looking for {len(emails_to_move)} unique emails in {source_match_sheet_name}")
            
            # Separate matching sheet rows
            for row in source_match_data:
                if len(row) > email_column_index and row[email_column_index] in emails_to_move:
                    match_rows_to_move.append(row)
                else:
                    match_rows_to_keep.append(row)
            
            self.logger.info(f"Found {len(match_rows_to_move)} matching rows to move, {len(match_rows_to_keep)} rows to keep in {source_match_sheet_name}")
            
            # Process in batches for appending (to avoid rate limits)
            total_processed = 0
            
            # Append rows to target sheets in batches
            for batch_start in range(0, len(rows_to_move), self.batch_size):
                batch_end = min(batch_start + self.batch_size, len(rows_to_move))
                batch_rows = rows_to_move[batch_start:batch_end]
                
                self.logger.info(f"Appending batch {batch_start//self.batch_size + 1}: rows {batch_start+1} to {batch_end}")
                
                # Append batch to target sheet
                self.append_rows_batch(spreadsheet_id, target_sheet_name, batch_rows)
                total_processed += len(batch_rows)
                
                # Small delay to avoid rate limiting
                if batch_end < len(rows_to_move):
                    time.sleep(0.5)
            
            # Append matching rows to target match sheet in batches
            for batch_start in range(0, len(match_rows_to_move), self.batch_size):
                batch_end = min(batch_start + self.batch_size, len(match_rows_to_move))
                batch_rows = match_rows_to_move[batch_start:batch_end]
                
                self.logger.info(f"Appending matching batch: rows {batch_start+1} to {batch_end}")
                
                # Append batch to target match sheet
                self.append_rows_batch(spreadsheet_id, target_match_sheet_name, batch_rows)
                
                # Small delay to avoid rate limiting
                if batch_end < len(match_rows_to_move):
                    time.sleep(0.5)
            
            # Now rewrite the source sheets with only the rows to keep
            self.logger.info(f"Rewriting {source_sheet_name} with {len(rows_to_keep)} rows (removing {len(rows_to_move)} rows)")
            self.clear_and_rewrite_sheet(spreadsheet_id, source_sheet_name, rows_to_keep)
            
            time.sleep(1)  # Brief pause between operations
            
            self.logger.info(f"Rewriting {source_match_sheet_name} with {len(match_rows_to_keep)} rows (removing {len(match_rows_to_move)} rows)")
            self.clear_and_rewrite_sheet(spreadsheet_id, source_match_sheet_name, match_rows_to_keep)
            
            # Final verification
            self.logger.info(f"\n=== Final Summary ===")
            self.logger.info(f"Moved {len(rows_to_move)} rows from {source_sheet_name} to {target_sheet_name}")
            self.logger.info(f"Moved {len(match_rows_to_move)} rows from {source_match_sheet_name} to {target_match_sheet_name}")
            self.logger.info(f"Remaining rows in {source_sheet_name}: {len(rows_to_keep)}")
            self.logger.info(f"Remaining rows in {source_match_sheet_name}: {len(match_rows_to_keep)}")
            
            return True
            
        except Exception as e:
            self.logger.error(f"Error in process_sheets: {str(e)}")
            return False


# Example usage
def main():
    # Replace with your service account credentials file path
    CREDENTIALS_FILE = "data/url-to-email-445616-cebe4868914f.json"
    
    # Replace with your Google Sheet URL or ID
    SHEET_URL = "https://docs.google.com/spreadsheets/d/1FFZ0c_SgpAcYgLU2gtVx4IFCFhOcQqf8DRdsarhCd_I/edit?gid=1108351937#gid=1108351937"
    
    # Initialize the mover with batch size of 20
    mover = GoogleSheetRowMover(CREDENTIALS_FILE, batch_size=20)
    
    # Extract spreadsheet ID from URL
    spreadsheet_id = mover.extract_spreadsheet_id(SHEET_URL)
    
    # List all available sheets to verify the correct names
    mover.list_all_sheets(spreadsheet_id)
    
    # Define the column indices (0-based)
    TARGET_COLUMN_INDEX = 22
    EMAIL_COLUMN_INDEX = 4
    
    # Use the actual sheet names as seen in the log output
    SOURCE_SHEET = "Sheet2"  # The sheet with "INSERT A RANDOM VALUE HERE" rows
    TARGET_SHEET = "Sheet3"  # Where to move those rows
    SOURCE_MATCH_SHEET = "Sheet1"  # The sheet with matching email rows
    TARGET_MATCH_SHEET = "Sheet4"  # Where to move those matching rows
    
    # Process the sheets
    mover.process_sheets(
        spreadsheet_id=spreadsheet_id,
        target_column_index=TARGET_COLUMN_INDEX,
        email_column_index=EMAIL_COLUMN_INDEX,
        source_sheet_name=SOURCE_SHEET,
        target_sheet_name=TARGET_SHEET,
        source_match_sheet_name=SOURCE_MATCH_SHEET,
        target_match_sheet_name=TARGET_MATCH_SHEET
    )

In [2]:
main()

2025-09-09 03:44:15,601 - INFO - file_cache is only supported with oauth2client<4.0.0
2025-09-09 03:44:16,690 - INFO - Available sheets in this spreadsheet:
2025-09-09 03:44:16,690 - INFO -   - 'Sheet1' (ID: 0)
2025-09-09 03:44:16,690 - INFO -   - 'Sheet2' (ID: 1108351937)
2025-09-09 03:44:16,691 - INFO -   - 'Sheet3' (ID: 1351164279)
2025-09-09 03:44:16,691 - INFO -   - 'Sheet4' (ID: 849645612)
2025-09-09 03:44:17,309 - INFO - Available sheets in this spreadsheet:
2025-09-09 03:44:17,309 - INFO -   - 'Sheet1' (ID: 0)
2025-09-09 03:44:17,310 - INFO -   - 'Sheet2' (ID: 1108351937)
2025-09-09 03:44:17,310 - INFO -   - 'Sheet3' (ID: 1351164279)
2025-09-09 03:44:17,310 - INFO -   - 'Sheet4' (ID: 849645612)
2025-09-09 03:44:17,944 - INFO - Found sheet ID for 'Sheet2': 1108351937 (type: <class 'int'>)
2025-09-09 03:44:18,656 - INFO - Found sheet ID for 'Sheet1': 0 (type: <class 'int'>)
2025-09-09 03:44:18,657 - INFO - Using sheet IDs: Sheet2=1108351937, Sheet1=0
2025-09-09 03:44:21,321 - INF